In [64]:
from bs4 import BeautifulSoup
import requests
import json
from datetime import datetime

In [ ]:
job_title = "frontend developer"
location = "Canada"
SOURCE = "LinkedIn"
#SOURCE = "Indeed"

headers = {
    "User-Agent": "Mozilla/5.0"
}

In [ ]:
job_list = []

for start in range(0, 100, 25):   # scrape 4 pages
    url = f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?keywords={job_title}&location={location}&start={start}"
    
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    jobs = soup.find_all("li")

    for job in jobs:
        base_card = job.find("div", {"class": "base-card"})
        if not base_card:
            continue

        # Extract the job ID from the URN
        job_id = base_card.get("data-entity-urn").split(":")[3]

        job_url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"

        job_response = requests.get(job_url, headers=headers)
        job_soup = BeautifulSoup(job_response.text, "html.parser")

        # --- Title ---
        title_tag = job_soup.find("h2")
        title = title_tag.text.strip() if title_tag else None

        # --- Company ---
        company_tag = job_soup.find("a", {"class": "topcard__org-name-link"})
        if not company_tag:
            company_tag = job_soup.find("span", {"class": "topcard__flavor"})
        company = company_tag.text.strip() if company_tag else None

        # --- Description ---
        description_tag = job_soup.find("div", {"class": "show-more-less-html__markup"})
        description = description_tag.text.strip() if description_tag else None

        # --- Source & Scraping date ---
        source = SOURCE
        scraped_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        job_data = {
            "title":        title,
            "company":      company,
            "description":  description,
            "source":       source,
            "scraped_at":   scraped_at
        }

        job_list.append(job_data)

print(f"Total jobs scraped: {len(job_list)}")

In [ ]:
# Load existing data if file exists
try:
    with open("jobs_dataanalyst.json", "r", encoding="utf-8") as f:
        existing_data = json.load(f)
except (FileNotFoundError, json.JSONDecodeError):
    existing_data = []

# Append new jobs
existing_data.extend(job_list)

# Write back
with open("jobs_dataanalyst.json", "w", encoding="utf-8") as f:
    json.dump(existing_data, f, indent=4, ensure_ascii=False)

print(f"Added {len(job_list)} jobs. Total now: {len(existing_data)}")